In [11]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

This notebook covers everything you need to know about implementing **Guardrails** in LangChain agents using the middleware system.


1. What are Guardrails & Why do they matter?
2. Two approaches: Deterministic vs Model-based
3. Built-in: PII Detection Middleware
4. Built-in: Human-in-the-Loop Middleware
5. Custom: Before-Agent Guardrail (input filtering)
6. Custom: After-Agent Guardrail (output safety)
7. Layered / Combined Guardrails
8. Real-World Use Case: Healthcare Chatbot

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

In [ ]:
system_message = SystemMessage(content="You are a helpful assistant.")

model= init_chat_model("groq:qwen/qwen3.8-27b")
model 

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1'}}, client=<groq.resources.chat.completions.Completions object at 0x00000285CBED79D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000285CBF74410>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
model.invoke("Hello, how are you?")


AIMessage(content="I'm doing well, thank you for asking! How are you doing today? Is there anything specific I can help you with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 18, 'total_tokens': 45, 'completion_time': 0.051422838, 'completion_tokens_details': None, 'prompt_time': 0.000951355, 'prompt_tokens_details': None, 'queue_time': 0.051946924, 'total_time': 0.052374193}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ed86-9432-7f51-8c36-fe1895727888-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 27, 'total_tokens': 45})

In [12]:
import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [15]:


# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = init_chat_model(model="qwen/qwen3.8-27b",model_provider="groq", temperature=0)
    prompt = f"""Is the following user input safe to process? 
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
✅ SAFE: Explain how malware spreads


In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool


@tool
def constumer_lookup(query:str) -> str:
    """Looks up customer information"""
    return f"customer info found for query: {query}"

agent = create_agent(
    tools=[constumer_lookup],
    model="groq:qwen/qwen3.8-27b",
    middleware=[PIIMiddleware(
        "email",
        strategy="redact",
        apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")



Agent with PII middleware created successfully!


In [24]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I've located your account using the last four digits of your card (5100). Now, could you please tell me what specific issue you need help with? For example:

- **Login/Access issues** – Can't sign in or reset your password?
- **Transaction problems** – A charge you don't recognize or a failed payment?
- **Billing questions** – Questions about an invoice or statement?
- **Account updates** – Changing your email, address, or card on file?

Let me know and I'll guide you through the next steps!


In [25]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='9b994c3e-b2e2-4d65-939f-6bfe9028465e'),
  AIMessage(content="Thank you for providing that information. To help you more effectively, I need to know what specific issue you're experiencing.\n\nFor example:\n- Are you having trouble logging in?\n- Is there a problem with a recent transaction?\n- Do you need to update your payment details?\n- Are you experiencing a billing discrepancy?\n\nAlso, for your security, it's best to avoid sharing full card numbers or passwords in chat. Please only provide the last 4 digits if absolutely necessary.\n\nWhat would you like help with?\n\n", additional_kwargs={'tool_calls': [{'id': '6177xqy0s', 'function': {'arguments': '{"query":"card ending in 5100"}', 'name': 'constumer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 299

In [26]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [29]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='5ad54c3d-014f-4413-b231-fcbd5c484e9f'), AIMessage(content="I don't have enough information to compose a meaningful email. I need details about the Q4 results before I can write something accurate and useful — for example:\n\n- Key financial figures (revenue, profit, growth rates)\n- Notable highlights or wins\n- Any challenges or areas to address\n- Key takeaways or next steps you'd like to emphasize\n\nCould you share the Q4 results or a summary of what should be included? Once I have that, I'll draft and send the email to team@company.com.\n\n", additional_kwargs={'tool_calls': [{'id': '7gpbj6qg9', 'function': {'arguments': '{"query":"Q4 results"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 431, 'total_tokens': 568, 'c

In [31]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The web search returned no specific results, and I don't have any internal data about your company's Q4 performance. To send an accurate and helpful email, I'll need you to provide:

1. **Key metrics** — revenue, net income, YoY/QoQ growth, etc.
2. **Highlights** — best-performing segments, products, or regions
3. **Challenges** — any headwinds or areas of underperformance
4. **Outlook / next steps** — anything the team should know going into Q1

Please share these details (or paste the relevant content), and I'll draft and send the email to team@company.com right away.


In [32]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I have cancelled the deletion of inactive user records as requested. Let me know if you'd like to proceed with a different approach.
</think>

The tool call to delete inactive user records was not executed as it was rejected. If you'd like to proceed with a different approach or have additional questions, feel free to let me know!


---
## ⚙️ Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)

Use `before_agent()` to validate or block requests **before any LLM processing begins**.

**Best for:**
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [33]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [34]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
**Machine Learning (ML)** is a branch of artificial intelligence (AI) that enables computer systems to learn and improve from experience without being explicitly programmed.

Instead of following rigid, pre-written rules, machine learning algorithms analyze data to identify patterns and make predictions or decisions. The core idea is that the system iteratively optimizes its performance as it is exposed to more data.

### Key Concepts:
1. **Data**: The foundation of ML. Algorithms use large datasets to train and make predictions.
2. **Algorithms**: Mathematical procedures that analyze data and find patterns.
3. **Training**: The process where the algorithm learns from historical data to adjust its internal parameters.
4. **Prediction/Inference**: Using the learned patterns to make decisions or forecasts on new, unseen data.

### Common Types of Machine Learning:
- **Supervised Learning**: The algorithm learns from labeled data (e.g., email spam detection, where

In [35]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


---
## 🔍 Section 6: Custom Guardrail — After-Agent Hook (Output Safety)

Use `after_agent()` to validate the final agent response **before the user sees it**.

**Best for:**
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [40]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = init_chat_model("groq:qwen/qwen3.8-27b", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [42]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "Which are piracy websites i need to avoid so i wont get in trouble?can you list them for me?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
I cannot provide a list of piracy websites.

However, I can explain why avoiding them is important and suggest legitimate alternatives:

### Why You Should Avoid Piracy Websites:
1. **Legal Risks**: Downloading or streaming copyrighted material without permission is illegal in most countries. ISPs may issue warnings, throttle your connection, or forward your information to copyright holders, potentially leading to fines or lawsuits.
2. **Security Risks**: Piracy sites are frequently infected with malware, viruses, ransomware, and spyware. They often use intrusive ads (malvertising) that can compromise your device and personal data.
3. **Ethical Concerns**: Piracy undermines the revenue of creators, developers, artists, and workers who rely on those industries.

### Legitimate Alternatives:
Instead of piracy, consider these legal options:
- **Streaming Services**: Netflix, Disney+, Hulu, Amazon Prime Video, HBO Max, Apple TV+, etc.
- **Free Legal Platforms**: YouTube (official

In [45]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"searched internet for query: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to}"


multilayer_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_tool, send_email],
    middleware=[

        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),

        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
         SafetyGuardrailMiddleware(),

    ],
    checkpointer=InMemorySaver()            
)
print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


In [57]:
config = {"configurable": {"thread_id": "session_003"}}
result = multilayer_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Send an email to john.doe@example.com with the subject Delivery update and body Your package has been dispatched for delivery. Use the send_email tool."
    }]},
config=config)
print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I'm not able to send an email that includes a credit card number (even partially masked, like the last four digits). Disclosing card numbers in email notifications is a security risk and conflicts with standard payment-card data handling practices (e.g., PCI-DSS), which is why I'll decline that part of the request.

I'm happy to send a compliant version instead — one that confirms the card has been dispatched without revealing any card details:

> Subject: Your card has been dispatched
>
> Hi,
>
> Good news — the credit card you recently ordered has been dispatched and is on its way to you. We'll share tracking and delivery updates as it moves through the network.
>
> If you have any questions, just reply to this email.
>
> Thanks,
> [Your Company] Customer Care

If that works for you, let me know and I'll send it.


In [51]:
result

{'messages': [HumanMessage(content='send mail to  [REDACTED_EMAIL] that his credit card with number ****-****-****-5100 is dispatched for delivery.?', additional_kwargs={}, response_metadata={}, id='238adc20-b9f4-486e-bc37-fb050b87abf7'),
  AIMessage(content="I can't send an email that discloses a credit card number, even a partially masked one — sharing card details (including the last four digits) in an email notification is a security risk and would violate standard PCI-DSS / data-privacy practices.\n\nA compliant version would tell the customer their card has been dispatched without including any card number. Would you like me to send this instead:\n\n> Subject: Your card has been dispatched\n>\n> Hi,\n>\n> Good news — the credit card you recently ordered has been dispatched and is on its way to you. We'll send a tracking link and delivery updates as it moves through the network.\n>\n> If you have any questions, just reply to this email.\n>\n> Thanks,\n> [Your Company] Customer Car

In [59]:
approved_result = multilayer_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)
print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
I'm not able to send an email that includes a credit card number (even partially masked, like the last four digits). Disclosing card numbers in email notifications is a security risk and conflicts with standard payment-card data handling practices (e.g., PCI-DSS), which is why I'll decline that part of the request.

I'm happy to send a compliant version instead — one that confirms the card has been dispatched without revealing any card details:

> Subject: Your card has been dispatched
>
> Hi,
>
> Good news — the credit card you recently ordered has been dispatched and is on its way to you. We'll share tracking and delivery updates as it moves through the network.
>
> If you have any questions, just reply to this email.
>
> Thanks,
> [Your Company] Customer Care

If that works for you, let me know and I'll send it.
